# Housekeeping

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Input, Dense, Lambda
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K
import tensorflow_datasets as tfds
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer, TFBertModel
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from torch.utils.data import DataLoader, Dataset, random_split
from torch import nn, optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report



In [2]:
from google.colab import files

uploaded = files.upload()

Saving fake_job_postings.csv to fake_job_postings.csv


In [3]:
df = pd.read_csv("fake_job_postings.csv")
df.head(5)

,job_id,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent
0,1,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0
1,2,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0
2,3,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,Valor Services provides Workforce Solutions th...,"Our client, located in Houston, is actively se...",Implement pre-commissioning and commissioning ...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0
3,4,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life thro...,THE COMPANY: ESRI – Environmental Systems Rese...,"EDUCATION: Bachelor’s or Master’s in GIS, busi...",Our culture is anything but corporate—we have ...,0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0
4,5,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,SpotSource Solutions LLC is a Global Human Cap...,JOB TITLE: Itemization Review ManagerLOCATION:...,QUALIFICATIONS:RN license in the State of Texa...,Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0


In [4]:
sum(df['fraudulent'] == 1)

866

# Data Cleaning

In [5]:
df.columns

Index(['job_id', 'title', 'location', 'department', 'salary_range',
       'company_profile', 'description', 'requirements', 'benefits',
       'telecommuting', 'has_company_logo', 'has_questions', 'employment_type',
       'required_experience', 'required_education', 'industry', 'function',
       'fraudulent'],
      dtype='object')

In [6]:
import re
import string
import spacy
import pandas as pd

# Fill NA with empty space
df.fillna(' ', inplace=True)

# Concatenate all text features into one big text
df['text'] = df['title'] + " " + df['company_profile'] + " " + df['description'] + " " + df['requirements'] + " " + df['benefits']

# Define the clean_text function
def clean_text(text):
    '''Make text lowercase, remove text in square brackets, remove links, remove punctuation
    and remove words containing numbers.'''
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

# Apply the clean_text function to the 'text' column
df['text'] = df['text'].apply(lambda x: clean_text(x))

# Data Cleanups
df['text'] = df['text'].str.replace('\n', '')
df['text'] = df['text'].str.replace('\r', '')
df['text'] = df['text'].str.replace('\t', '')
df['text'] = df['text'].apply(lambda x: re.sub(r'[0-9]', '', x))
df['text'] = df['text'].apply(lambda x: re.sub(r'[/(){}\[\]\|@,;.:-]', ' ', x))
df['text'] = df['text'].apply(lambda s: s.lower() if type(s) == str else s)
df['text'] = df['text'].str.replace('  ', ' ')

# Remove Stop words
nlp = spacy.load("en_core_web_sm")
df['text'] = df['text'].apply(lambda x: ' '.join([word for word in x.split() if nlp.vocab[word].is_stop == False]))

# Drop unnecessary columns
df = df[['text', 'fraudulent']]

# Display the modified DataFrame
print(df.head(3))



                                                text  fraudulent
0  marketing intern weve created groundbreaking a...           0
1  customer service cloud video production second...           0
2  commissioning machinery assistant cma valor se...           0


In [7]:
df.iloc[0]['text']

'marketing intern weve created groundbreaking awardwinning cooking site support connect celebrate home cooks need placewe editorial business engineering team focused technology find new better ways connect people specific food interests offer superb highly curated information food cooking attract talented home cooks contributors country publish wellknown professionals like mario batali gwyneth paltrow danny meyer partnerships foods market random named best food website james beard foundation iacp featured new york times npr pando daily techcrunch today showwere located chelsea new york city fastgrowing james beard awardwinning online food community crowdsourced curated recipe hub currently interviewing parttime unpaid interns work small team editors executives developers new york city headquartersreproducing andor repackaging existing content number partner sites huffington post yahoo buzzfeed content management systemsresearching blogs websites provisions affiliate programassisting da

# Modeling

In [8]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size = 0.20, random_state = 1)


# Data Balancing

In [9]:
df_good_ads = df[df['fraudulent'] == 0]
df_fraud_ads = df[df['fraudulent'] == 1]

In [10]:
len(df_fraud_ads)

866

# 1 to 1 Balancing

In [11]:
# Randomly sample from df1
df_good_ads_sampled = df_good_ads.sample(n = 866, random_state=42)

# Concatenate
one_to_one_df = pd.concat([df_good_ads_sampled, df_fraud_ads], axis=0)

In [12]:
one_to_one_df

,text,fraudulent
5230,sem coordinator modern online travel agency fo...,0
14112,senior data scientist growing successful start...,0
3168,junior web marketing specialist η atnet commun...,0
14804,new product development project leader time pe...,0
5809,data intern retail apparel analysis build soft...,0
...,...,...
17827,student positions parttime fulltime student po...,1
17828,sales associate learn earn executive level inc...,1
17829,android developer infullmobile sp z oo mobile ...,1
17830,payroll clerk job descriptionwe seeking time p...,1


In [13]:
X_train, X_test, y_train, y_test = train_test_split(one_to_one_df['text'], one_to_one_df['fraudulent'], test_size = 0.20, random_state = 1)


# 2 to 1 Balancing

In [14]:
# Randomly sample from df1
df_good_ads_sampled = df_good_ads.sample(n = 866 * 2, random_state=42)

# Concatenate
two_to_one_df = pd.concat([df_good_ads_sampled, df_fraud_ads], axis=0)

In [15]:
two_to_one_df

,text,fraudulent
5230,sem coordinator modern online travel agency fo...,0
14112,senior data scientist growing successful start...,0
3168,junior web marketing specialist η atnet commun...,0
14804,new product development project leader time pe...,0
5809,data intern retail apparel analysis build soft...,0
...,...,...
17827,student positions parttime fulltime student po...,1
17828,sales associate learn earn executive level inc...,1
17829,android developer infullmobile sp z oo mobile ...,1
17830,payroll clerk job descriptionwe seeking time p...,1


GloVe

In [16]:
import gensim.downloader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a classifier (e.g., Logistic Regression)
classifier = LogisticRegression()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

[==================================================] 100.0% 66.0/66.0MB downloaded

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3395
           1       0.82      0.05      0.09       181

    accuracy                           0.95      3576
   macro avg       0.88      0.52      0.53      3576
weighted avg       0.94      0.95      0.93      3576



In [17]:
import gensim.downloader
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a classifier (e.g., Logistic Regression)
classifier = LogisticRegression()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.87      0.83       356
           1       0.64      0.53      0.58       164

    accuracy                           0.76       520
   macro avg       0.72      0.70      0.71       520
weighted avg       0.75      0.76      0.75       520



In [18]:
import gensim.downloader
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Linear Support Vector Machine)
classifier = LinearSVC()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98      3395
           1       1.00      0.04      0.08       181

    accuracy                           0.95      3576
   macro avg       0.98      0.52      0.53      3576
weighted avg       0.95      0.95      0.93      3576



In [19]:
import gensim.downloader
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Linear Support Vector Machine)
classifier = LinearSVC()

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84       356
           1       0.66      0.55      0.60       164

    accuracy                           0.77       520
   macro avg       0.74      0.71      0.72       520
weighted avg       0.76      0.77      0.77       520



In [20]:
import gensim.downloader
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Random Forest)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       1.00      0.43      0.60       181

    accuracy                           0.97      3576
   macro avg       0.99      0.71      0.79      3576
weighted avg       0.97      0.97      0.97      3576



In [21]:
import gensim.downloader
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Random Forest)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.92       356
           1       0.88      0.73      0.79       164

    accuracy                           0.88       520
   macro avg       0.88      0.84      0.85       520
weighted avg       0.88      0.88      0.88       520



In [22]:
import gensim.downloader
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Gradient Boosting)
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98      3395
           1       0.91      0.33      0.48       181

    accuracy                           0.96      3576
   macro avg       0.94      0.66      0.73      3576
weighted avg       0.96      0.96      0.96      3576



In [23]:
import gensim.downloader
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., Gradient Boosting)
classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88       356
           1       0.79      0.66      0.72       164

    accuracy                           0.84       520
   macro avg       0.82      0.79      0.80       520
weighted avg       0.83      0.84      0.83       520



In [24]:
import gensim.downloader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., k-Nearest Neighbors)
classifier = KNeighborsClassifier(n_neighbors=5)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99      3395
           1       0.89      0.60      0.72       181

    accuracy                           0.98      3576
   macro avg       0.94      0.80      0.85      3576
weighted avg       0.97      0.98      0.97      3576



In [25]:
import gensim.downloader
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0]  # Use TF-IDF vector if no words are present in GloVe model
    return sum(vectorized_words) / len(vectorized_words)

# Convert documents to GloVe vectors
X_train_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train]
X_test_glove = [document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test]

# Choose a different classifier (e.g., k-Nearest Neighbors)
classifier = KNeighborsClassifier(n_neighbors=5)

# Train the classifier
classifier.fit(X_train_glove, y_train)

# Predictions
y_pred = classifier.predict(X_test_glove)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.93      0.91       356
           1       0.84      0.76      0.79       164

    accuracy                           0.88       520
   macro avg       0.87      0.84      0.85       520
weighted avg       0.88      0.88      0.88       520



In [26]:
import gensim.downloader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['fraudulent'], test_size=0.2, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0].toarray()  # Use TF-IDF vector if no words are present in GloVe model
    return np.mean(vectorized_words, axis=0)

# Convert documents to GloVe vectors
X_train_glove = np.array([document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train])
X_test_glove = np.array([document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test])

# Pad the GloVe sequences
max_len = X_train_glove.shape[1]  # Use the number of features as the maximum length
X_train_pad = pad_sequences(X_train_glove, maxlen=max_len)
X_test_pad = pad_sequences(X_test_glove, maxlen=max_len)

# Build the CNN model
embedding_dim = 50  # Dimension of the word embeddings
model = Sequential()
model.add(Embedding(input_dim=X_train_glove.shape[1], output_dim=embedding_dim, input_length=max_len))
model.add(Conv1D(128, 5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"\nTest Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Predictions
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Epoch 1/5
179/179 [==============================] - 13s 11ms/step - loss: 0.2281 - accuracy: 0.9517 - val_loss: 0.1993 - val_accuracy: 0.9493
Epoch 2/5
179/179 [==============================] - 1s 8ms/step - loss: 0.2031 - accuracy: 0.9528 - val_loss: 0.2007 - val_accuracy: 0.9493
Epoch 3/5
179/179 [==============================] - 1s 8ms/step - loss: 0.1985 - accuracy: 0.9528 - val_loss: 0.2040 - val_accuracy: 0.9493
Epoch 4/5
179/179 [==============================] - 1s 6ms/step - loss: 0.1992 - accuracy: 0.9528 - val_loss: 0.1994 - val_accuracy: 0.9493
Epoch 5/5
112/112 [==============================] - 0s 3ms/step - loss: 0.2025 - accuracy: 0.9494

Test Loss: 0.2025, Test Accuracy: 0.9494
112/112 [==============================] - 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97      3395
           1       0.00      0.00      0.00       181

    accuracy                           0.95      3

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [27]:
import gensim.downloader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Assuming 'df' is your DataFrame with 'text' and 'fraudulent' columns

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(two_to_one_df['text'], two_to_one_df['fraudulent'], test_size = 0.20, random_state=42)

# Convert text data to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Load the pre-trained GloVe model
embed_vectors = gensim.downloader.load('glove-wiki-gigaword-50')

# Function to convert a document to a vector representation using GloVe
def document_to_vector(document, model, tfidf_vectorizer):
    words = document.split()
    vectorized_words = [model[word] for word in words if word in model.key_to_index]
    if not vectorized_words:
        return tfidf_vectorizer.transform([document])[0].toarray()  # Use TF-IDF vector if no words are present in GloVe model
    return np.mean(vectorized_words, axis=0)

# Convert documents to GloVe vectors
X_train_glove = np.array([document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_train])
X_test_glove = np.array([document_to_vector(doc, embed_vectors, tfidf_vectorizer) for doc in X_test])

# Pad the GloVe sequences
max_len = X_train_glove.shape[1]  # Use the number of features as the maximum length
X_train_pad = pad_sequences(X_train_glove, maxlen=max_len)
X_test_pad = pad_sequences(X_test_glove, maxlen=max_len)

# Build the CNN model
embedding_dim = 50  # Dimension of the word embeddings
model = Sequential()
model.add(Embedding(input_dim=X_train_glove.shape[1], output_dim=embedding_dim, input_length=max_len))
model.add(Conv1D(128, 5, activation='relu'))
model.add(GlobalMaxPooling1D())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.2)

# Evaluate the model
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"\nTest Loss: {loss:.4f}, Test Accuracy: {accuracy:.4f}")

# Predictions
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Epoch 1/5
26/26 [==============================] - 3s 34ms/step - loss: 0.6534 - accuracy: 0.6552 - val_loss: 0.6401 - val_accuracy: 0.6611
Epoch 2/5
26/26 [==============================] - 0s 12ms/step - loss: 0.6429 - accuracy: 0.6625 - val_loss: 0.6405 - val_accuracy: 0.6611
Epoch 3/5
26/26 [==============================] - 0s 10ms/step - loss: 0.6423 - accuracy: 0.6625 - val_loss: 0.6395 - val_accuracy: 0.6611
Epoch 4/5
26/26 [==============================] - 0s 10ms/step - loss: 0.6415 - accuracy: 0.6625 - val_loss: 0.6393 - val_accuracy: 0.6611
Epoch 5/5
17/17 [==============================] - 0s 6ms/step - loss: 0.6199 - accuracy: 0.6846

Test Loss: 0.6199, Test Accuracy: 0.6846
17/17 [==============================] - 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.68      1.00      0.81       356
           1       0.00      0.00      0.00       164

    accuracy                           0.68       520
   mac

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
